# 🚀 Airflow Parallel Execution + Task Instance (TI) in XCom

---

# 🧠 1. Parallel Execution

Parallel execution means multiple tasks run at the same time.

Example:
task1 >> [task2, task3, task4]

---

# 🧠 2. Task Instance (TI)

TI = Task Instance
Represents one execution of a task

Used for XCom:
- ti.xcom_push()
- ti.xcom_pull()

---

# ⚙️ 3. Classic Example

```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def push_data(ti):
    ti.xcom_push(key="value", value=10)

def task_a(ti):
    val = ti.xcom_pull(task_ids="push_task", key="value")
    print(val)

with DAG("parallel_xcom", start_date=datetime(2024,1,1), schedule="@daily", catchup=False) as dag:

    push = PythonOperator(task_id="push_task", python_callable=push_data)
    a = PythonOperator(task_id="task_a", python_callable=task_a)

    push >> a
```

---

# ⚡ 4. Decorator Example

```python
from airflow.decorators import dag, task
from datetime import datetime

@dag(start_date=datetime(2024,1,1), schedule="@daily", catchup=False)
def example():

    @task
    def push():
        return 10

    @task
    def consume(val):
        print(val)

    consume(push())

dag = example()
```

---

# 🎯 Summary

- Parallel = multiple tasks at same time
- TI = execution unit
- XCom uses TI to share data

---

# 🔥 End


# 🚀 Airflow Branching — Classic vs Decorator

---

# 🧠 What is Branching?

Branching allows conditional execution of tasks in Airflow.

Only selected downstream tasks run, others are skipped.

---

# 🔥 Classic Example (with DAG)

```python
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from datetime import datetime

def choose_path():
    return "task_a"

def task_a():
    print("A")

def task_b():
    print("B")

with DAG("branching_classic", start_date=datetime(2024,1,1), schedule="@daily", catchup=False) as dag:

    branch = BranchPythonOperator(
        task_id="branch",
        python_callable=choose_path
    )

    t1 = PythonOperator(task_id="task_a", python_callable=task_a)
    t2 = PythonOperator(task_id="task_b", python_callable=task_b)

    branch >> [t1, t2]
```

---

# ⚡ Decorator Example

```python
from airflow.decorators import dag, task
from datetime import datetime

@dag(start_date=datetime(2024,1,1), schedule="@daily", catchup=False)
def branching():

    @task.branch
    def choose():
        return "task_a"

    @task
    def task_a():
        print("A")

    @task
    def task_b():
        print("B")

    choose() >> [task_a(), task_b()]

dag = branching()
```

---

# 🔄 Parameter-Based Branching

```python
@task.branch
def choose(params=None):
    return "task_a" if params["type"] == "A" else "task_b"
```

---

# ⚠️ Notes

- Return valid task_id
- Non-selected tasks are skipped
- Use trigger_rule for joins

---

# 🎯 Summary

Branching enables conditional workflows using classic or decorator approaches.

---

# 🔥 End
